In [1]:
import logging
import csv
import re
import os
import time
import random
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

logging.basicConfig(
    format='%(asctime)s %(levelname)s:%(message)s',
    level=logging.INFO
)

class Crawler:

    BASE_URL = "https://www.trondheim.kommune.no/aktuelt/kunngjoring-arealplan/"

    ALLOWED_PREFIXES = [
        BASE_URL + "offentlig-ettersyn/",
        BASE_URL + "igangsatt-planarbeid/",
        BASE_URL + "vedtatte-planer/",
    ]

    BLOCKED_PREFIXES = [
        BASE_URL + "arkiv-vedtatte-planer/",
        BASE_URL + "arkiv-igangsatt-planarbeid/",
        BASE_URL + "arkiv-planer-kunngjort/",
        BASE_URL + "arkiv-andre-planer/",
    ]

    FILE_EXTENSIONS = (".pdf", ".doc", ".docx", ".xls", ".xlsx")

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/119.0.0.0 Safari/537.36"
            ),
            "Accept-Language": "no-NO,no;q=0.9,en-US;q=0.8,en;q=0.7",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Referer": self.BASE_URL
        })
        self.visited_urls = set()
        self.urls_to_visit = [self.BASE_URL]
        self.relevant_links = set()
        self.results = []
        self.downloaded_files = set()

    # ----- Hent HTML -----
    def download_url(self, url):
        response = self.session.get(url, timeout=15)
        response.raise_for_status()
        time.sleep(random.uniform(1, 3))
        return response.text

    # ----- Finn alle lenker på en side -----
    def get_linked_urls(self, url, html):
        soup = BeautifulSoup(html, "html.parser")
        for link in soup.find_all("a", href=True):
            href = link["href"]
            if href.startswith("/"):
                href = urljoin(url, href)
            if not href.startswith(self.BASE_URL):
                continue
            if any(href.startswith(blocked) for blocked in self.BLOCKED_PREFIXES):
                continue
            yield href

    def sanitize_filename(self, name):
        return re.sub(r'[\\/*?:"<>|]', "_", name)

    # ----- Ekstraher metadata -----
    def extract_page_data(self, html):
        soup = BeautifulSoup(html, "html.parser")
        h1 = soup.find("h1")
        title = h1.get_text(strip=True) if h1 else "Uten tittel"

        full_text = soup.get_text(separator="\n", strip=True)

        frist = None
        oppdatert = None

        frist_match = re.search(
            r"Frist for innspill\s*:?\s*([0-9]{1,2}\.[0-9]{1,2}\.[0-9]{4})",
            full_text, re.IGNORECASE
        )

        oppdatert_match = re.search(
            r"Sist oppdatert\s*:?\s*([0-9]{1,2}\.[0-9]{1,2}\.[0-9]{4})",
            full_text, re.IGNORECASE
        )

        if frist_match:
            frist = frist_match.group(1)

        if oppdatert_match:
            oppdatert = oppdatert_match.group(1)

        return title, frist, oppdatert

    # ----- Ekstraher kun rich-text -----
    def extract_content_text(self, html):
        soup = BeautifulSoup(html, "html.parser")

        content_div = soup.find("div", class_="rich-text")

        if not content_div:
            return None

        for tag in content_div(["script", "style", "noscript"]):
            tag.decompose()

        content_text = content_div.get_text(separator="\n", strip=True)

        # Valgfritt: begrens lengde
        if content_text:
            content_text = content_text[:15000]

        return content_text

    # ----- Finn dokumentlenker -----
    def extract_file_links(self, page_url, html):
        soup = BeautifulSoup(html, "html.parser")
        file_links = []
        for link in soup.find_all("a", href=True):
            href = link["href"]
            if href.startswith("/"):
                href = urljoin(page_url, href)
            if href.lower().endswith(self.FILE_EXTENSIONS):
                file_links.append(href)
        return file_links

    # ----- Last ned filer -----
    def download_file(self, file_url, folder):
        if file_url in self.downloaded_files:
            return

        os.makedirs(folder, exist_ok=True)

        filename = file_url.split("/")[-1]
        filepath = os.path.join(folder, filename)

        try:
            response = self.session.get(file_url, timeout=30)
            response.raise_for_status()

            with open(filepath, "wb") as f:
                f.write(response.content)

            self.downloaded_files.add(file_url)
            logging.info(f"Downloaded: {filename}")

            time.sleep(random.uniform(1, 2))

        except Exception as e:
            logging.error(f"Failed downloading {file_url}: {e}")

    # ----- Crawl en side -----
    def crawl(self, url):
        html = self.download_url(url)

        for link in self.get_linked_urls(url, html):

            if any(link.startswith(prefix) for prefix in self.ALLOWED_PREFIXES):
                self.relevant_links.add(link)

            if link not in self.visited_urls:

                if any(link.startswith(prefix) for prefix in self.ALLOWED_PREFIXES):
                    logging.info(f"Processing plan page: {link}")

                    try:
                        page_html = self.download_url(link)

                        title, frist, oppdatert = self.extract_page_data(page_html)
                        content_text = self.extract_content_text(page_html)

                        folder_name = self.sanitize_filename(title)
                        folder_path = os.path.join("documents", folder_name)

                        file_links = self.extract_file_links(link, page_html)

                        for file_url in file_links:
                            self.download_file(file_url, folder_path)

                        self.results.append({
                            "url": link,
                            "title": title,
                            "frist_for_innspill": frist,
                            "sist_oppdatert": oppdatert,
                            "documents_downloaded": len(file_links),
                            "content": content_text
                        })

                    except Exception as e:
                        logging.error(f"Failed processing {link}: {e}")

                if link not in self.urls_to_visit:
                    self.urls_to_visit.append(link)

    # ----- Start crawler -----
    def run(self):
        while self.urls_to_visit:
            url = self.urls_to_visit.pop(0)

            if url in self.visited_urls:
                continue

            logging.info(f"Crawling: {url}")

            try:
                self.crawl(url)
            except Exception as e:
                logging.error(f"Failed: {url} ({e})")

            self.visited_urls.add(url)

        self.save_results()
        self.save_relevant_links()

    # ----- Lagre metadata -----
    def save_results(self):
        with open("relevante_lenker.csv", "w", newline="", encoding="utf-8") as csvfile:
            fieldnames = [
                "url",
                "title",
                "frist_for_innspill",
                "sist_oppdatert",
                "documents_downloaded",
                "content"
            ]

            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

            for row in self.results:
                writer.writerow(row)

        logging.info(f"Saved {len(self.results)} plans to relevante_lenker.csv")

    # ----- Lagre alle relevante lenker -----
    def save_relevant_links(self):
        with open("alle_relevante_lenker.txt", "w", encoding="utf-8") as f:
            for link in sorted(self.relevant_links):
                f.write(link + "\n")

        logging.info(f"Saved {len(self.relevant_links)} links to alle_relevante_lenker.txt")


if __name__ == "__main__":
    crawler = Crawler()
    crawler.run()

2026-03-18 10:13:07,575 INFO:Crawling: https://www.trondheim.kommune.no/aktuelt/kunngjoring-arealplan/
2026-03-18 10:13:08,940 INFO:Processing plan page: https://www.trondheim.kommune.no/aktuelt/kunngjoring-arealplan/igangsatt-planarbeid/igangsatt-planarbeid-for-oppheving-av-reguleringsplaner/
2026-03-18 10:13:10,955 INFO:Downloaded: kart--og-plandokumenter-18870826.pdf
2026-03-18 10:13:12,679 INFO:Downloaded: kart--og-plandokumenter-18970405.pdf
2026-03-18 10:13:17,206 INFO:Downloaded: kart--og-plandokumenter-r0306.pdf
2026-03-18 10:13:19,112 INFO:Downloaded: kart--og-plandokumenter-r1098b.pdf
2026-03-18 10:13:21,553 INFO:Downloaded: kart--og-plandokumenter-r0203.pdf
2026-03-18 10:13:24,256 INFO:Downloaded: kart--og-plandokumenter-r1098c.pdf
2026-03-18 10:13:26,609 INFO:Downloaded: kart--og-plandokumenter-r0298.pdf
2026-03-18 10:13:28,757 INFO:Downloaded: kart--og-plandokumenter-r0112.pdf
2026-03-18 10:13:30,372 INFO:Processing plan page: https://www.trondheim.kommune.no/aktuelt/kunng